# SYDE 556/750 &mdash; Practice Notebook 5
## Symbols and the Semantic Pointer Architecture

**Prepares you for: Test 5.**

This notebook introduces the building blocks of the **Semantic Pointer Architecture (SPA)**: high-dimensional random vectors as symbols, bundling (superposition), binding (circular convolution), unbinding, and cleanup memory. The first four sections are pure `numpy` from scratch; the last builds a neural SPA model in Nengo. Much of the content here mirrors the Neuromatch NeuroAI W2D2 tutorial (Tutorials 1 and 3) on neurosymbolic methods.

**How to use it**
- **No solutions are released.** Each part ends with an *Expected result* checkpoint so you can tell whether you are on track.
- Cells marked &#x270D; are for written answers.

**Contents**
1. High-dimensional vector symbols and similarity
2. Bundling: representing a composite object
3. Binding and unbinding with circular convolution
4. Cleanup memory
5. A neural SPA: visual input and working memory


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nengo

rng = np.random.default_rng(556)


# 1. High-dimensional vector symbols and similarity

In SPA we represent concepts (CAT, DOG, RED, $\ldots$) as random unit vectors in a high-dimensional space. Two properties make this work: (i) random high-dimensional unit vectors are nearly orthogonal (ie., they have small dot products with each other) so different symbols are distinguishable, and (ii) a vector is similar to itself, so we can recognise a noisy copy by comparing to the original.


**a) Random unit vectors.** Write a function `random_unit_vector(d, rng)` that draws a random unit vector in $d$ dimensions (sample from a $d$-dim Gaussian and normalize). Generate 1000 pairs of independent random unit vectors at each dimension $d \in \{16, 64, 256, 1024\}$, compute the dot product within each pair, and plot the *distribution* of those dot products (one histogram per $d$, on the same axes or in a small grid).

&#x1F4CC; In SPA, "high-dimensional" typically means $d \ge 256$.


In [ ]:
# random_unit_vector(d, rng): draw d-dim Gaussian, divide by its norm.
# For d in [16, 64, 256, 1024], generate 1000 independent pairs and histogram the dot products.
# TODO


*Expected:* All four distributions are centred on $0$, but the width shrinks as $d$ grows. At $d = 1024$ the dot product of two random unit vectors is almost always within $\pm 0.1$ of zero.

**b) Why does similarity shrink as $d$ grows?** State (with one sentence and one equation) how the standard deviation of the dot product of two random unit vectors in $d$ dimensions depends on $d$.

&#x270D; *Your answer here.*


**c) Recognising a noisy copy.** Pick a single random unit vector $\vec v$ in $d = 256$. Add Gaussian noise of standard deviation $\sigma$ for $\sigma \in \{0.0, 0.1, 0.3, 1.0\}$, re-normalize, and compute the dot product of the noisy vector with $\vec v$. Compare to the dot product of $\vec v$ with a *different* random unit vector. What is the noise level at which you can no longer reliably tell "noisy $\vec v$" from "a different vector"?


In [ ]:
# For sigma in [0, 0.1, 0.3, 1.0]: compute <noisy(v), v> and <other, v>; print both.
# TODO


# 2. Bundling: representing a composite object

The first SPA operation is **bundling** (superposition): just add vectors together, optionally normalising the result. A bundled vector is *similar to* each of its constituents (in the sense of positive dot product), so we can ask ``is X in this bundle?'' by computing $\langle B, X\rangle$.


**a) Bundled shape.** Make a vocabulary of three shape vectors `CIRCLE`, `SQUARE`, `TRIANGLE` and three colour vectors `RED`, `GREEN`, `BLUE`, all in $d = 256$. Compute the bundle $B = \mathrm{CIRCLE} + \mathrm{SQUARE} + \mathrm{TRIANGLE}$ (normalised to unit length). Plot the similarity of $B$ to each of the six vocabulary items as a bar chart.


In [ ]:
# Build the 6-item vocabulary; bundle the three shapes; bar-plot similarity to all 6 items.
# TODO


*Expected:* CIRCLE, SQUARE, TRIANGLE each have similarity $\approx 1/\sqrt{3}$ with the bundle; the colours have similarities very close to $0$.

**b) How many items can you bundle?** Take a vocabulary of $K$ random unit vectors in $d = 256$, bundle all of them (normalised), and compute the similarity of the bundle with each constituent. Repeat for $K \in \{2, 4, 8, 16, 32, 64\}$. Plot the *mean* similarity of a bundle's constituents (and the standard deviation across constituents) vs. $K$. In one sentence, what does the trend tell you about how many concepts can be held in superposition before they become indistinguishable from noise?

&#x270D; *Your one-sentence interpretation here.*


In [ ]:
# For K in [2, 4, 8, 16, 32, 64], bundle K random items in d=256, measure mean similarity with constituents.
# Plot mean (and std) vs K.
# TODO


# 3. Binding and unbinding with circular convolution

To represent *structured* knowledge (role-filler pairs like ``the colour is red'') we need a second operation: **binding**. The standard SPA binding operation is **circular convolution**:
$$\vec A \circledast \vec B \;=\; \mathcal{F}^{-1}\bigl\{ \mathcal{F}\{\vec A\}\, \cdot\, \mathcal{F}\{\vec B\} \bigr\},$$
where $\mathcal{F}$ is the FFT and $\cdot$ is element-wise multiplication. Binding produces a new vector that is approximately *orthogonal* to both inputs, i.e., the bound vector "looks different" from either of its parts.

To **unbind** (extract one factor given the other) we use the approximate inverse $\vec A^{\,-1} \approx \tilde{\vec A}$, where $\tilde{\vec A}$ is the *involution* of $\vec A$: keep the first component, reverse the rest. Then
$$\bigl(\vec A \circledast \vec B\bigr) \circledast \tilde{\vec A} \;\approx\; \vec B.$$


**a) Implement** the binding and unbinding operations:
- `bind(a, b)`: circular convolution of two vectors of the same length, computed via FFT.
- `involution(a)`: returns $\tilde{\vec a}$.
- `unbind(c, a)`: returns $c \circledast \tilde{\vec a}$.

Sanity check by binding two random unit vectors and confirming that the result is approximately orthogonal to each input.


In [ ]:
# Implement bind(a, b), involution(a), unbind(c, a).
# Sanity: <a, bind(a,b)> and <b, bind(a,b)> should both be near 0 (in high d).
# TODO


*Expected:* in $d = 256$ both sanity dot products are within $\pm 0.1$ of $0$.

**b) Role-filler pair.** Build a vocabulary with `COLOR`, `SHAPE`, `RED`, `CIRCLE` (random unit vectors in $d = 256$). Construct the "colourful shape" representation
$$\vec S \;=\; \mathrm{COLOR} \circledast \mathrm{RED} \;+\; \mathrm{SHAPE} \circledast \mathrm{CIRCLE}.$$


In [ ]:
# Construct S as the sum of the two bindings above.
# TODO


**c) Query.** Extract the colour by unbinding $\vec S$ with `COLOR`: compute $\vec S \circledast \widetilde{\mathrm{COLOR}}$ and compare (by dot product) to the four vocabulary items. Repeat for the query "what shape?" using `SHAPE`.


In [ ]:
# Compute the unbindings and report similarity to all four vocabulary items.
# TODO


*Expected:* the COLOR query returns a vector most similar to `RED` (others are near zero); the SHAPE query returns a vector most similar to `CIRCLE`.

**d) Why is unbinding only approximate?** In one or two sentences, explain why the unbinding result is *similar to* but *not exactly* the original filler. (Hint: think about what the bundle $\vec S$ looks like to the unbinding operation.)

&#x270D; *Your answer here.*


# 4. Cleanup memory

The result of an unbinding is a *noisy* copy of the original vector (because the bundle had other terms mixed in). A **cleanup memory** stores a known vocabulary and `cleans up' any noisy input to the closest stored vector. This is what makes our VSA act like symbolic computation.


**a) Implement cleanup.** Write `cleanup(query, vocab_names, vocab_vectors)` that returns the name and vector of the vocabulary item with the highest dot product to `query`. Run cleanup on the COLOR and SHAPE queries from &sect;3c and confirm it returns `RED` and `CIRCLE`.


In [ ]:
# cleanup(query, vocab_names, vocab_vectors) -> (best_name, best_vector)
# TODO


**b) Cleanup vs. dimensionality.** For dimensions $d \in \{8, 16, 32, 64, 128, 256\}$, build a 20-item filler vocabulary and 4 role vectors $\mathrm{ROLE}_1, \ldots, \mathrm{ROLE}_4$. Encode 100 random 4-slot role-filler bundles of the form $\sum_{k=1}^{4} \mathrm{ROLE}_k \circledast \mathrm{FILLER}_{i_k}$ (each filler drawn uniformly at random from the vocabulary, without replacement across slots), unbind each bundle with $\widetilde{\mathrm{ROLE}_1}$, run cleanup, and measure how often it recovers the correct filler for slot 1. Plot accuracy vs. $d$.


In [ ]:
# Sweep d; for each d build 20-item vocab, encode 100 random role-filler bundles,
# query for the colour, cleanup, and measure accuracy.  Plot accuracy vs d.
# TODO


*Expected:* accuracy is poor at low $d$ and approaches 1 as $d$ grows past $\sim 256$.

**c) Discussion.** Why does cleanup accuracy improve as $d$ increases? Tie your answer back to what you found in &sect;1 about the similarity of random vectors.

&#x270D; *Your answer here.*


# 5. A neural VSA: Visual input and working memory

`nengo_spa` is a high-level interface for building SPA models in Nengo. It includes the VSA operators. For this use case, it defines a `Vocabulary` (the set of random vectors we just built by hand) and `State` modules (population ensembles that hold a semantic pointer). All of the operations above (bundle, bind, cleanup) become connections between these modules.

For this section, use $d = 32$ (small enough to simulate quickly, large enough for the dynamics to work).


**a) Visual input.** Build an SPA network with a single 32-D `State` populated by an input that alternates between the vectors `CAT` and `DOG` every 400 ms for the first 1.6 s, then returns to the zero vector. Show a similarity plot (similarity of the state's output to each of `CAT`, `DOG` and a third filler `BEAR`) over 2 s. In a sentence or two, explain what the similarity plot is showing.

&#x1F40D; The `nengo_spa.similarity(...)` function computes the dot product of a probe trace with every vector in a vocabulary, all in one call.


In [ ]:
# Build a 32-D SPA model with one State driven by a Node that alternates CAT/DOG.
# Probe the state, plot the similarity to CAT, DOG, BEAR over 2 s.
# TODO


**b) Add a working memory.** Add a second `State` whose feedback gain is `1` (so it integrates its input). Connect the visual input to the working memory through a small input transform (e.g., $0.1$). Show a similarity plot of the working memory's output over 2 s.


In [ ]:
# Add a second State with feedback=1; connect visual input -> memory with a small transform.
# Plot similarity of the memory state to CAT, DOG, BEAR.
# TODO


**c) Long-run memory.** Run the model for 20 s with the same alternating input over the first 1.6 s and zero input afterwards. Plot both states' similarity traces. Comment on what you see: how long does the memory state preserve a recognisable representation of `CAT` and `DOG`? Is there any sign of a *primacy* or *recency* effect (one of CAT or DOG dominating)?

&#x270D; *Your interpretation here (1-2 sentences).*


In [ ]:
# Run 20 s; plot the similarity for both states.
# TODO


---
*End of Practice Notebook 5.* Test 5 will assume you can do everything above by hand or from scratch.
